Setting up environment: modules imports and pyplot.rcParams update

In [ ]:
from src.lib import ThinFilmDatabase as db
from src.lib import fitVRH

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib import ticker

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12,
    "text.latex.preamble": r"\usepackage[T1]{fontenc}",
    "text.latex.preamble": r"\usepackage{xcolor}"
})

High temperature conductivity fit.

Mott variable range hopping model in three dimensions:
$$
\sigma(T) =
\sigma_{0,M}
\exp\left[-(T_{0,M}/T)^{1/4}\right],
$$
where
* $\sigma_{0,M}$ -- a pre-exponential factor, 
* $T_{0,M}$ -- the characteristic temperature.

In [ ]:
def conductivity_ht():
    """Mott 3D VRH fit"""
    mott = lambda a1, t1, t: a1*t**(-1/2)*np.exp(-(t1/t)**0.25)
    df = db.load_df()
    fig = plt.figure(figsize=(3.2, 3), tight_layout=True, dpi=200)
    ax = fig.add_gridspec(1, 1, right=.98, left=0.18, top=0.85, bottom=0.16).subplots()
    for sample in df['TL10_10':'TL10_30'].index:
        t_min, t_max = db.ht_range(sample)
        data = db.read_conductivity(sample).loc[t_max+1:t_min-1]
        temp, cond = data.index.to_numpy(), data.to_numpy()
        par = db.plt_params(sample)
        ax.plot(np.log10(temp), np.log10(cond), 
            par['m'][1:], label=par['label'], mec=par['c'], mfc='w', c=par['c'], ms=6)
        # fitting
        a1, t1, *_ = fitVRH(temp, cond, 0.25)
        ax.plot(np.log10(temp), np.log10(mott(a1, t1, temp)), '--', c=par['c'])
        # print(f'Fitting results [{i}]: {coef}')
    ax.set_ylabel(r'$\log_{10}\sigma$, where $\sigma$ [($\Omega$cm)$^{-1}$]')
    ax.set_xlabel(r'$\log_{10}T$, where $T$ [K]')
    ax.set_ylim(-8, 0)
    ax.set_xlim(np.log10(140), np.log10(300))
    xticks = [280, 240, 200, 160]
    ax.set_xticks(np.log10(xticks))
    # # Second x-axis
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(np.log10(xticks))
    ax2.set_xticklabels(xticks)
    ax2.set_xlabel(r'$T$ [K]')
    ax.legend(loc="upper right", title=r'$x$', ncol=1, bbox_to_anchor=(.48, 1.05, 0, 0),
              frameon=False, handletextpad=0.1)
    ax.tick_params(axis='both', which='both', direction='in', pad=5)
    ax2.tick_params(axis='both', which='both', direction='in', pad=5)
    ax.xaxis.set_major_formatter('{x:.2f}')
    plt.savefig('output/conductivity_ht.pdf', dpi=600)


if __name__ == '__main__':
    conductivity_ht()